# Live demo · inverting TRIBE v2 in front of the panel

**VIT Project-I · Abhranil Sarkar (23BML0077), Tanisha Choudhuri (23BML0081) · Guide: Dr. Sharmila Nageswaran**

This notebook runs the project's core experiment live on the real 177 M-parameter model, in about 10 minutes on a free T4 GPU:

| Step | What you show | Time |
|---|---|---|
| 1–2 | Install, download the frozen TRIBE v2 weights | ≈4 min (do this before the viva) |
| 3 | The model and the differentiable re-entry point | 20 s |
| 4 | **Live inversion**: three starts, plots update as it runs | ≈3 min |
| 5 | **Ridge test** and **sensitivity probe**, forward passes only | 30 s |
| 6 | Target vs predicted response on the cortex | 20 s |
| 7 | Measurement noise: recovery degrades, identification survives | ≈1 min |

`Runtime → Change runtime type → T4 GPU`, then run the cells one at a time. No Llama access and no Hugging Face token are needed: the inversion starts after TRIBE v2's feature extractors.

## Step 1 · Install and clone (then **Runtime → Restart session**)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install uv
!uv pip -q install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip -q install --system "transformers<4.57" "torchaudio==2.6.0" "numpy<2.1" h5py scikit-learn
!git clone -q https://github.com/Ivxn404/semantic-decoding-tribe-inversion /content/repo || (cd /content/repo && git pull -q)
print('INSTALL DONE - now click Runtime > Restart session, then continue with Step 2')

## Step 2 · Settings and frozen weights

In [ ]:
import os, sys
os.environ['PROJECT_ROOT'] = '/content/repo'
os.environ['TRIBE_WEIGHTS'] = '/content/repo/weights'
os.chdir('/content/repo/scripts'); sys.path.insert(0, '/content/repo/scripts')
sys.path = [p for p in sys.path if not p.endswith('tribev2')]
from huggingface_hub import hf_hub_download
for f in ['config.yaml', 'best.ckpt']:
    hf_hub_download('facebook/tribev2', f, local_dir='/content/repo/weights')
print('weights ready')

## Step 3 · The forward operator f(x)

We re-enter TRIBE v2 after feature aggregation: x is the T × 1 152 stimulus representation (text | audio | video slots of 384), f(x) is the predicted response at 20 484 cortical vertices × 100 output steps. Every weight is frozen; only x will be optimised.

In [ ]:
import math, time, warnings, numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
warnings.filterwarnings('ignore')
from load_tribe import load, make_fwd
from roi_inversion import make_truth
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net, _ = load(); net = net.to(dev); fwd = make_fwd(net)
print(f'device: {dev}')
print(f'parameters: {sum(p.numel() for p in net.parameters())/1e6:.2f} M, all frozen: {all(not p.requires_grad for p in net.parameters())}')
H = 1152
x_true = make_truth(H, seed=0).to(dev)                     # known ground truth, 32 time steps x 1 152
with torch.no_grad():
    y = fwd(x_true)                                        # the target response
print('x shape', tuple(x_true.shape), '->  f(x) shape', tuple(y.shape))
xg = x_true.clone().requires_grad_(True); fwd(xg).square().mean().backward()
print(f'gradient reaches x: norm {xg.grad.norm():.3g}, zero entries {(xg.grad == 0).sum().item()}')

## Step 4 · Live inversion from three starts

Adam, learning rate 5×10⁻², cosine schedule, exactly as in the report. The three starts are the paper's: Gaussian s.d. 0.01, Gaussian s.d. 1.0, uniform on [−1, 1]. Watch the left panel fall by orders of magnitude while the middle panel stalls below 1, and the right panel show the starts landing in different places.

In [ ]:
STEPS = 1500          # raise to 4000+ if you have time; the report uses 16 000-32 000
g = torch.Generator().manual_seed(999)
starts = {'Gaussian 0.01': 0.01 * torch.randn(1, 32, H, generator=g),
          'Gaussian 1.0': 1.0 * torch.randn(1, 32, H, generator=g),
          'Uniform [-1, 1]': torch.rand(1, 32, H, generator=g) * 2 - 1}
names = list(starts); K = len(names)
x = torch.cat(list(starts.values())).to(dev).requires_grad_(True)
opt = torch.optim.Adam([x], lr=5e-2); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
Y = y.expand(K, -1, -1)
cosT = lambda a: F.cosine_similarity(a, x_true[0], dim=1).mean().item()
flat = lambda a, b: F.cosine_similarity(a.reshape(-1), b.reshape(-1), dim=0).item()
hist = {'step': [], 'mse': [], 'cos': [], 'pair': []}
cols = ['#1b5fc1', '#c2255c', '#2b8a3e']
t0 = time.time()
for i in range(STEPS + 1):
    if i % 50 == 0:
        with torch.no_grad():
            p = fwd(x)
            hist['step'].append(i); hist['mse'].append([F.mse_loss(p[k], y[0]).item() for k in range(K)])
            hist['cos'].append([cosT(x[k]) for k in range(K)]); hist['pair'].append(min(flat(x[a], x[b]) for a, b in ((0, 1), (0, 2), (1, 2))))
        clear_output(wait=True)
        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        for k in range(K):
            ax[0].semilogy(hist['step'], [m[k] for m in hist['mse']], color=cols[k], label=names[k])
            ax[1].plot(hist['step'], [c[k] for c in hist['cos']], color=cols[k])
        ax[2].plot(hist['step'], hist['pair'], color='k')
        ax[0].set_title('prediction error (log)'); ax[1].set_title('recovery of the true x (cosine)'); ax[2].set_title('lowest agreement between starts')
        ax[1].axhline(1, ls='--', c='grey'); ax[1].set_ylim(0, 1); ax[2].set_ylim(-0.1, 1.05); ax[0].legend()
        for a in ax: a.set_xlabel('step'); a.set_xlim(0, STEPS)
        plt.tight_layout(); plt.show()
        e0 = hist['mse'][0][0] / hist['mse'][-1][0]
        print(f"step {i}/{STEPS} · {time.time()-t0:.0f} s · error down {e0:,.0f}x · recovery " + ' / '.join(f'{c:.3f}' for c in hist['cos'][-1]) + f" · agreement {hist['pair'][-1]:.3f}")
    if i == STEPS: break
    opt.zero_grad(); loss = F.mse_loss(fwd(x), Y) * K; loss.backward(); opt.step(); sch.step()

## Step 5 · Ridge test and sensitivity probe (forward passes only)

No optimiser is involved, so neither result can be an optimiser artefact.

* **Ridge test.** Walk in a straight line from start 1 to start 2 and measure the prediction error against the target along the way. If the two solutions were points of one flat valley (a null space), the error would stay at the endpoints' level all the way. A rise in the middle means a ridge separates two equally good fits: the problem is non-convex.
* **Sensitivity probe.** How strongly f responds along the difference d and along the true error, compared with 100 random directions.

In [ ]:
with torch.no_grad():
    a, b = x[0:1].detach(), x[1:2].detach(); d = b - a
    path = np.linspace(0, 1, 11)
    err = [F.mse_loss(fwd(a + s * d), y).item() for s in path]
plt.figure(figsize=(7, 3.2)); plt.semilogy(path, err, 'o-', c='#c2255c')
plt.xlabel('position on the straight path (0 = start 1, 1 = start 2)'); plt.ylabel('prediction error'); plt.title('straight path between two solutions'); plt.tight_layout(); plt.show()
peak = max(err[1:-1]); ends = max(err[0], err[-1])
print(f'endpoints {err[0]:.2e} and {err[-1]:.2e}; worst point on the path {peak:.2e} = {peak/ends:.1f}x the worse endpoint')
print('-> a ridge separates the two fits (non-convex)' if peak > 1.5 * ends else '-> no clear ridge at this budget; the report uses 14 000 steps and fit-matched starts')
gr = torch.Generator().manual_seed(11)
with torch.no_grad():
    fa = fwd(a)
    def sens(v, eps=1e-2):
        v = v / v.norm(); return ((fwd(a + eps * v) - fa).norm() / eps).item()
    s_d, s_e = sens(d), sens(a - x_true)
    s_r = np.array([sens(torch.randn(d.shape, generator=gr).to(dev)) for _ in range(100)])
plt.figure(figsize=(8, 3.2)); plt.hist(s_r, bins=25, color='grey', label='100 random directions')
plt.axvline(s_d, c='#c2255c', lw=3, label='along d (start 1 to start 2)'); plt.axvline(s_e, c='#d9480f', lw=3, label='along the true error')
plt.xlabel('sensitivity ||f(x+eps v) - f(x)|| / eps'); plt.legend(); plt.tight_layout(); plt.show()
print(f'random median is {np.median(s_r)/s_d:.1f}x the sensitivity along d and {np.median(s_r)/s_e:.1f}x along the error (report: 1.9x at 16 000 steps)')

## Step 6 · Target and prediction on the cortex (fsaverage5, 20 484 vertices)

In [ ]:
from nilearn import datasets, plotting
fs = datasets.fetch_surf_fsaverage('fsaverage5')
with torch.no_grad():
    pt = fwd(x[0:1])[0, :, 50].cpu().numpy(); tt = y[0, :, 50].cpu().numpy()
vmax = np.percentile(np.abs(tt), 99)
fig, ax = plt.subplots(1, 4, figsize=(18, 4), subplot_kw={'projection': '3d'})
for j, (vals, title) in enumerate([(tt, 'target'), (pt, 'prediction from x-hat')]):
    for h, hemi in enumerate(['left', 'right']):
        v = vals[:10242] if hemi == 'left' else vals[10242:]
        plotting.plot_surf_stat_map(fs[f'infl_{hemi}'], v, hemi=hemi, bg_map=fs[f'sulc_{hemi}'], vmax=vmax, colorbar=False,
                                    axes=ax[j * 2 + h], title=f'{title} · {hemi}', threshold=vmax * 0.2)
plt.show()
print(f'correlation of target and prediction across vertices: {np.corrcoef(tt, pt)[0, 1]:.4f}   (recovery of x: {cosT(x[0]):.3f})')

## Step 7 · Measurement noise: identification survives, recovery does not

Invert a noisy target (noise s.d. = 50% of the signal s.d.) and ask whether x-hat still picks out the right time step among the 32.

In [ ]:
torch.manual_seed(0)
yn = y + 0.5 * y.std() * torch.randn_like(y)
xn = (0.01 * torch.randn(1, 32, H, generator=torch.Generator().manual_seed(999))).to(dev).requires_grad_(True)
o = torch.optim.Adam([xn], lr=5e-2); sc = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=800)
for i in range(800):
    o.zero_grad(); F.mse_loss(fwd(xn), yn).backward(); o.step(); sc.step()
with torch.no_grad():
    S = F.normalize(xn[0], dim=1) @ F.normalize(x_true[0], dim=1).T
    ident = (S.argmax(1) == torch.arange(32, device=dev)).float().mean().item()
print(f'recovery cosine with noise: {cosT(xn[0]):.3f}  ·  time step identified correctly: {ident*100:.0f}% (chance {100/32:.1f}%)')

### What this showed
1. The fit improves by orders of magnitude while recovery saturates: **encoding accuracy does not certify decoding**.
2. Starts that fit equally well land in different places, separated by a **ridge**, not a flat null space: the inverse is **non-convex**.
3. The remaining error lies along directions the model barely responds to: **anisotropic ill-conditioning**.
4. With noise, the moment is still **identified** while the representation is not **reconstructed** — the same pattern as on real fMRI.

Real-fMRI analysis (LeBel et al. 2023, held-out story): `notebooks/RealData_TextAudio.ipynb`. Full synthetic suite: `notebooks/Demo_and_Synthetic.ipynb`.